# 02-1. 極限 — 動かして確かめる

📖 解説: [`../01_limits.md`](../01_limits.md)

## このノートで触るもの
1. sin(x)/x の x→0 の極限を数値的に観察
2. SymPy で厳密な極限計算
3. e の定義の確認
4. floor 関数の不連続性を可視化
5. 数値微分のための極限

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (01_limits.md)](../01_limits.md)

In [ ]:
import numpy as np
import sympy as sp
import jax.numpy as jnp
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)
from ipywidgets import interact, FloatLogSlider

%matplotlib inline

## 1. sin(x)/x の x→0 を数値的に観察

$$
\lim_{x \to 0} \frac{\sin(x)}{x} = 1
$$

対応するコード: `np.sin(x) / x`

In [ ]:
xs = np.array([1.0, 0.1, 0.01, 0.001, 0.0001, 0.00001])
ys = np.sin(xs) / xs
for x, y in zip(xs, ys):
    print(f'x = {x:.6f}  →  sin(x)/x = {y:.10f}')
print('\n→ 1 にどんどん近づくのが見える')

## 2. SymPy で厳密に

$$
\lim_{x \to \infty} \frac{1}{x} = 0, \qquad \lim_{x \to \infty}\left(1 + \frac{1}{x}\right)^x = e
$$

In [ ]:
x = sp.Symbol('x')
print('lim_{x→0} sin(x)/x  =', sp.limit(sp.sin(x)/x, x, 0))
print('lim_{x→∞} 1/x       =', sp.limit(1/x, x, sp.oo))
print('lim_{x→∞} (1+1/x)^x =', sp.limit((1 + 1/x)**x, x, sp.oo), '  ← e そのもの!')
print('lim_{x→0} (1+x)^(1/x)=', sp.limit((1 + x)**(1/x), x, 0), '  ← e のもう1つの定義')

## 3. e の定義を数値的に確かめる

$$
e = \lim_{n \to \infty}\left(1 + \frac{1}{n}\right)^n \approx 2.71828\dots
$$

対応するコード: `(1 + 1/n) ** n`

In [ ]:
import math

for n in [1, 2, 12, 365, 10_000, 1_000_000, 100_000_000]:
    val = (1 + 1/n) ** n
    print(f'n = {n:>11}  →  (1 + 1/n)^n = {val:.10f}')
print(f'\n真の e          = {math.e:.10f}')

## 4. 不連続関数 (floor) の挙動を可視化

floor 関数 $\lfloor x \rfloor$ は整数値で不連続:

$$
\lim_{x \to 1^-} \lfloor x \rfloor = 0, \qquad \lim_{x \to 1^+} \lfloor x \rfloor = 1
$$

In [ ]:
x = np.linspace(-3, 3, 1000)
y_floor = np.floor(x)
y_smooth = np.sin(x)  # 比較用に連続関数

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].step(x, y_floor, where='post', color='red')
ax[0].set_title('floor(x): 整数値ごとにジャンプ → 不連続')
ax[0].grid(True, alpha=0.3)

ax[1].plot(x, y_smooth, color='blue')
ax[1].set_title('sin(x): 滑らか → 連続')
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 数値微分のための極限 — h を変えてみる

中央差分 (誤差 $O(h^2)$):

$$
f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}
$$

In [ ]:
# f(x) = x^2 の x = 2 における微分は 2x = 4 のはず
def f(x):
    return x ** 2

x0 = 2.0
for h in [1.0, 0.1, 0.01, 0.001, 1e-6, 1e-10, 1e-15]:
    df_approx = (f(x0 + h) - f(x0)) / h
    print(f'h = {h:.0e}  →  近似値 = {df_approx:.10f}')
print('\n真の値 = 4.0')
print('→ h が小さすぎると浮動小数点誤差で逆に悪化することに注意')

## 6. スライダーで h を動かして観察

In [ ]:
def plot_secant(h: float) -> None:
    """f(x)=x^2 のグラフと、x=2 における割線 (h で近似) を描く."""
    xs = np.linspace(0, 4, 100)
    ys = xs ** 2
    x0 = 2.0
    y0 = x0 ** 2
    x1 = x0 + h
    y1 = x1 ** 2
    slope = (y1 - y0) / h
    
    # 割線
    line_x = np.linspace(0, 4, 10)
    line_y = y0 + slope * (line_x - x0)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(xs, ys, 'b-', label='f(x) = x²')
    ax.plot(line_x, line_y, 'r--', label=f'割線 (傾き {slope:.4f})')
    ax.plot([x0, x1], [y0, y1], 'go', markersize=10)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'h = {h:.4f},  傾き ≈ {slope:.4f}  (真の微分 = 4)')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

interact(plot_secant, h=FloatLogSlider(value=1.0, base=10, min=-3, max=0.3, step=0.1, description='h'));

## まとめ
- `sin(x)/x` の x→0 は 1
- e は `lim (1+1/n)^n` で定義される (n=10⁸ で 8桁一致)
- floor は不連続、sin は連続
- 数値微分は h が小さすぎると壊れる (sweet spot は 1e-5 〜 1e-7)

## 次へ
→ [`02_derivatives.ipynb`](02_derivatives.ipynb) (微分)

解説 → [`../02_derivatives.md`](../02_derivatives.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [章 TOP](../README.md) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`02_derivatives.ipynb`](02_derivatives.ipynb) |